<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python for Finance, 3rd Edition
## Chapter 13 · Stochastics
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


Stochastic models describe how quantities such as prices, interest rates, or
risk factors evolve when randomness plays a central role. This chapter shows
how to work with pseudo-random numbers, simulate stochastic processes, and use
Monte Carlo methods for pricing and risk analysis in a way that integrates
smoothly with the tools from the previous chapters. You start with random
number generation and basic distributions, then move to discrete-time and
continuous-time processes, and finally use these building blocks to value
options by simulation. Throughout the chapter you work in the IPython REPL and
visualize key results with figures generated from scripts under
`code/figures/`.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

## Random Numbers and Distributions


Numerical work in stochastics begins with random numbers.


### Uniform and Normal Samples


Most simulation schemes ultimately depend on samples from the uniform
distribution on $[0, 1]$ or from the standard normal distribution.


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
mpl.style.use("seaborn-v0_8")
mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})
rng = np.random.default_rng(seed=42)
# Draw large samples from the uniform and standard normal distributions.
u = rng.random(100_000)
z = rng.standard_normal(100_000)

In [ ]:
u.mean(), u.std()

In [ ]:
z.mean(), z.std()

In [ ]:
# Plot the two empirical distributions in the same style used in the chapter
# figures.
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=False)
ax = axes[0]
ax.hist(u, bins=40, density=True, color="tab:blue", alpha=0.7)
ax.set_title("Uniform Samples on [0, 1)")
ax.set_xlabel("u")
ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.3)
ax = axes[1]
ax.hist(z, bins=40, density=True, color="tab:orange", alpha=0.7)
ax.set_title("Standard Normal Samples")
ax.set_xlabel("z")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### Sampling from Other Distributions


The same generator object also exposes methods for many other
distributions, including lognormal, Poisson, and multivariate normal.


## Simulation of Stochastic Processes


Random numbers become most useful once you combine them into paths of
stochastic processes.


### Discrete-Time Random Walk


A random walk is a simple process where each step adds a random increment to
the current value.


In [ ]:
n_steps = 250
n_paths = 5
shocks = rng.standard_normal((n_steps, n_paths))
# Scale and cumulate the increments to obtain discrete random-walk paths.
steps = shocks / np.sqrt(n_steps)
walk_increments = steps.cumsum(axis=0)
walk = np.vstack([np.zeros(n_paths), walk_increments])
walk[:3]

In [ ]:
# Recompute the random walk with the figure seed so the plotted paths match
# the chapter.
rng_walk = np.random.default_rng(seed=123)
n_steps_fig = 250
n_paths_fig = 5
shocks_fig = rng_walk.standard_normal((n_steps_fig, n_paths_fig))
steps_fig = shocks_fig / np.sqrt(n_steps_fig)
walk_fig = np.vstack([np.zeros(n_paths_fig), steps_fig.cumsum(axis=0)])
fig, ax = plt.subplots(figsize=(7.5, 4.0))
x_axis = np.arange(0, n_steps_fig + 1)
for j in range(n_paths_fig):
    ax.plot(x_axis, walk_fig[:, j], linewidth=1.0, alpha=0.9)
ax.set_title("Discrete-Time Random Walk Paths")
ax.set_xlabel("Step")
ax.set_ylabel("Level")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### Geometric Brownian Motion Paths


In the Black–Scholes model, an asset price $S_t$ follows a geometric Brownian
motion driven by a Brownian motion $W_t$:


In [ ]:
s0 = 100.0
mu = 0.05
sigma = 0.2
T = 1.0
n_steps = 252
dt = T / n_steps
n_paths = 20
shocks = rng.standard_normal((n_steps, n_paths))
# The discretized log return combines deterministic drift and random shocks.
log_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * shocks
log_paths = np.vstack([np.zeros(n_paths), log_returns.cumsum(axis=0)])
# Exponentiating the cumulative log returns maps the paths back to price space.
s_paths = s0 * np.exp(log_paths)

In [ ]:
# Recompute the GBM paths with the figure seed to reproduce the chapter plot.
rng_gbm = np.random.default_rng(seed=2027)
s0_fig = 100.0
mu_fig = 0.05
sigma_fig = 0.2
T_fig = 1.0
n_steps_fig = 252
n_paths_fig = 20
dt_fig = T_fig / n_steps_fig
shocks_fig = rng_gbm.standard_normal((n_steps_fig, n_paths_fig))
drift_fig = (mu_fig - 0.5 * sigma_fig**2) * dt_fig
log_returns_fig = (
    drift_fig + sigma_fig * np.sqrt(dt_fig) * shocks_fig
)
log_paths_fig = np.vstack(
    [np.zeros(n_paths_fig), log_returns_fig.cumsum(axis=0)]
)
s_paths_fig = s0_fig * np.exp(log_paths_fig)
fig, ax = plt.subplots(figsize=(7.5, 4.0))
x_axis = np.arange(0, n_steps_fig + 1)
ax.plot(x_axis, s_paths_fig, linewidth=1.0, alpha=0.8)
ax.set_title("Geometric Brownian Motion Paths")
ax.set_xlabel("Step")
ax.set_ylabel("Price")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### Terminal GBM Levels as Random Variables


Sometimes you only care about the terminal level $S_T$ of a GBM, not about the
full path.


In [ ]:
s0 = 100.0
r = 0.02
sigma = 0.2
T = 1.0
n_paths = 250_000
z = rng.standard_normal(n_paths)
# The first construction uses the analytical GBM log-price formula directly.
s_T_norm = s0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)
mean = np.log(s0) + (r - 0.5 * sigma**2) * T
std = sigma * np.sqrt(T)
# The second construction draws from a matching lognormal distribution.
s_T_logn = rng.lognormal(mean=mean, sigma=std, size=n_paths)

In [ ]:
# Overlay histograms of the two terminal-price simulation schemes.
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.hist(
    s_T_norm,
    bins=80,
    density=True,
    alpha=0.6,
    color="tab:blue",
    label="Standard-normal sampling",
)
ax.hist(
    s_T_logn,
    bins=80,
    density=True,
    alpha=0.4,
    color="tab:orange",
    label="Lognormal sampling",
)
ax.set_title("Terminal GBM Levels from Two Simulation Schemes")
ax.set_xlabel("Terminal price $S_T$")
ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

### Square-Root Diffusion (CIR) via Euler Scheme


Short-rate and variance models often use square-root diffusions of the
Cox–Ingersoll–Ross (CIR) form


In [ ]:
x0 = 0.05
kappa = 3.0
theta = 0.02
sigma = 0.1
T = 1.0
n_steps = 252
n_paths = 200_000
dt = T / n_steps
x = np.empty((n_steps + 1, n_paths))
x[0] = x0
for t in range(1, n_steps + 1):
    z = rng.standard_normal(n_paths)
    x_prev = x[t - 1]
    x[t] = x_prev + kappa * (theta - x_prev) * dt + sigma * np.sqrt(
        np.maximum(x_prev, 0.0)
    ) * np.sqrt(dt) * z
    # Truncate negative values to keep the simulated rates non-negative.
    x[t] = np.maximum(x[t], 0.0)

In [ ]:
# Show both the Euler-scheme maturity histogram and a small set of sample paths.
rng_cir = np.random.default_rng(seed=2027)
n_paths_hist = 200_000
n_paths_paths = 10
x_hist = np.empty((n_steps + 1, n_paths_hist))
x_hist[0] = x0
for t in range(1, n_steps + 1):
    z_hist = rng_cir.standard_normal(n_paths_hist)
    x_prev = x_hist[t - 1]
    drift = kappa * (theta - x_prev) * dt
    diffusion = sigma * np.sqrt(np.maximum(x_prev, 0.0))
    x_hist[t] = x_prev + drift + diffusion * np.sqrt(dt) * z_hist
    x_hist[t] = np.maximum(x_hist[t], 0.0)
x_paths = np.empty((n_steps + 1, n_paths_paths))
x_paths[0] = x0
for t in range(1, n_steps + 1):
    z_paths = rng_cir.standard_normal(n_paths_paths)
    x_prev = x_paths[t - 1]
    drift = kappa * (theta - x_prev) * dt
    diffusion = sigma * np.sqrt(np.maximum(x_prev, 0.0))
    x_paths[t] = x_prev + drift + diffusion * np.sqrt(dt) * z_paths
    x_paths[t] = np.maximum(x_paths[t], 0.0)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0))
ax = axes[0]
ax.hist(x_hist[-1], bins=80, density=True, color="tab:blue", alpha=0.7)
ax.set_title("CIR Euler Scheme: Values at Maturity")
ax.set_xlabel("x(T)")
ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.3)
ax = axes[1]
time_axis = np.linspace(0.0, T, n_steps + 1)
for j in range(n_paths_paths):
    ax.plot(time_axis, x_paths[:, j], linewidth=1.0, alpha=0.9)
ax.set_title("CIR Euler Scheme: Sample Paths")
ax.set_xlabel("Time")
ax.set_ylabel("x(t)")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### Exact Discretization of the Square-Root Diffusion


For the CIR process there exists an exact discretization in terms of a
noncentral chi-square distribution.


In [ ]:
df = 4.0 * theta * kappa / sigma**2
c = sigma**2 * (1.0 - np.exp(-kappa * dt)) / (4.0 * kappa)
x_exact = np.empty(n_paths)
x_exact[:] = x0
for _ in range(n_steps):
    nc = (
        4.0
        * kappa
        * np.exp(-kappa * dt)
        / (sigma**2 * (1.0 - np.exp(-kappa * dt)))
        * x_exact
    )
    x_exact = c * rng.noncentral_chisquare(df=df, nonc=nc, size=n_paths)

In [ ]:
# Plot the exact-discretization histogram at maturity.
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.hist(x_exact, bins=80, density=True, color="tab:green", alpha=0.7)
ax.set_title("CIR Exact Discretization: Values at Maturity")
ax.set_xlabel("x(T)")
ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## Monte Carlo Valuation of Options


With GBM paths in place, you can value derivatives by simulating the
underlying and averaging discounted payoffs.


### European Call Option under GBM


Under the risk-neutral measure, the stock price follows a GBM with drift equal
to the risk-free rate $r$.


In [ ]:
from math import exp, log, sqrt
r = 0.02
K = 100.0
T = 1.0
n_paths = 250_000
dt = T / n_steps
shocks = rng.standard_normal((n_steps, n_paths))
# Under the risk-neutral measure, the GBM drift becomes `r` instead of `mu`.
log_returns = (r - 0.5 * sigma**2) * dt + sigma * sqrt(dt) * shocks
log_paths = log_returns.cumsum(axis=0)
s_T = s0 * np.exp(log_paths[-1])
payoffs = np.maximum(s_T - K, 0.0)
discount = exp(-r * T)
price_mc = discount * payoffs.mean()
price_mc

In [ ]:
# Plot the discounted call-payoff distribution generated by the Monte Carlo
# simulation.
discounted_payoffs = discount * payoffs
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.hist(discounted_payoffs, bins=60, density=True, color="tab:blue", alpha=0.7)
ax.set_title("Discounted Monte Carlo Call Payoffs")
ax.set_xlabel("Discounted payoff")
ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### American Put Option via Least-Squares Monte Carlo


American options allow early exercise on a set of exercise dates.


In [ ]:
from math import erf
def norm_cdf(x: float) -> float:
    # `norm_cdf()` implements the standard normal CDF needed for the
    # Black-Scholes formula.
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))
def euro_put_bs(s0: float, K: float, r: float, sigma: float, T: float) -> float:
    if T <= 0.0:
        return max(K - s0, 0.0)
    d1 = (np.log(s0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    # `euro_put_bs()` provides the European benchmark across strikes.
    return K * np.exp(-r * T) * norm_cdf(-d2) - s0 * norm_cdf(-d1)
def lsm_american_put(
    s0: float,
    K: float,
    r: float,
    sigma: float,
    T: float,
    n_steps: int,
    n_paths: int,
    seed: int = 0,
) -> float:
    dt = T / n_steps
    rng_local = np.random.default_rng(seed)
    shocks = rng_local.standard_normal((n_steps, n_paths))
    log_returns = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * shocks
    log_paths = np.vstack([np.zeros(n_paths), log_returns.cumsum(axis=0)])
    # `S` stores simulated GBM price paths across all exercise dates.
    S = s0 * np.exp(log_paths)
    # `h` contains the intrinsic value of the put at each time and path.
    h = np.maximum(K - S, 0.0)
    V = h.copy()
    for t in range(n_steps - 1, 0, -1):
        in_the_money = h[t] > 0.0
        if not np.any(in_the_money):
            continue
        X = S[t, in_the_money]
        # Discount the next-step cash flows to approximate continuation values.
        Y = V[t + 1, in_the_money] * np.exp(-r * dt)
        # Use polynomial basis functions in the spot price for the regression.
        A = np.column_stack([np.ones_like(X), X, X**2])
        coeffs, *_ = np.linalg.lstsq(A, Y, rcond=None)
        continuation = A @ coeffs
        exercise = h[t, in_the_money]
        # Exercise when intrinsic value exceeds continuation value.
        exercise_now = exercise > continuation
        idx = np.where(in_the_money)[0][exercise_now]
        V[t, idx] = exercise[exercise_now]
        # Paths stop contributing future cash flows after exercise.
        V[t + 1 :, idx] = 0.0
    prices_0 = V[1] * np.exp(-r * dt)
    return float(prices_0.mean())

In [ ]:
# Compare European and American put prices across strikes using the chapter
# figure setup.
strikes = np.linspace(60.0, 140.0, 9)
euro_prices = np.array(
    [euro_put_bs(s0, strike, r, sigma, T) for strike in strikes]
)
american_prices = np.array(
    [
        lsm_american_put(
            s0, strike, r, sigma, T, 50, 75_000, seed=2029 + i
        )
        for i, strike in enumerate(strikes)
    ]
)
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(
    strikes,
    euro_prices,
    marker="o",
    linestyle="-",
    color="tab:blue",
    label="European put",
)
ax.plot(
    strikes,
    american_prices,
    marker="s",
    linestyle="--",
    color="tab:red",
    label="American put (LSM)",
)
ax.set_xlabel("Strike")
ax.set_ylabel("Option price")
ax.set_title("European vs. American Put Prices (Monte Carlo / LSM)")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

### Monte Carlo Convergence Behaviour


Monte Carlo estimators converge at a rate proportional to $1 / \sqrt{N}$,
where $N$ is the number of paths.


In [ ]:
def mc_call_price(n_paths: int, seed: int = 0) -> float:
    local_rng = np.random.default_rng(seed)
    shocks = local_rng.standard_normal((n_steps, n_paths))
    log_returns = (r - 0.5 * sigma**2) * dt + sigma * sqrt(dt) * shocks
    log_paths = log_returns.cumsum(axis=0)
    s_T = s0 * np.exp(log_paths[-1])
    payoffs = np.maximum(s_T - K, 0.0)
    return discount * float(payoffs.mean())
# Evaluate the estimator from 1,000 up to 1,000,000 paths.
grid = np.logspace(3, 6, num=12, dtype=int)
# Collect Monte Carlo estimates across a wide range of path counts.
estimates = [mc_call_price(int(n), seed=2027) for n in grid]

In [ ]:
# Add the Black-Scholes benchmark to visualize Monte Carlo convergence on a
# log scale.
d1 = (np.log(s0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
bs_price = s0 * norm_cdf(d1) - K * np.exp(-r * T) * norm_cdf(d2)
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(
    grid,
    estimates,
    marker="o",
    linestyle="-",
    color="tab:blue",
    label="MC estimate",
)
ax.axhline(
    bs_price,
    color="tab:red",
    linestyle="--",
    linewidth=1.25,
    label="Black-Scholes price",
)
ax.set_xscale("log")
ax.set_xlabel("Number of paths (log scale)")
ax.set_ylabel("Call price estimate")
ax.set_title("Convergence of Monte Carlo Call Price")
ax.legend(loc="best")
ax.grid(True, which="both", linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## Risk Measures


In addition to valuation, risk management is a key application of stochastic
simulation.


### Value-at-Risk and Expected Shortfall


Consider a single equity position valued at `s0` today.


In [ ]:
s0 = 100.0
r = 0.0
sigma = 0.25
T = 30.0 / 365.0
n_paths = 250_000
z = rng.standard_normal(n_paths)
# Simulate terminal prices over the 30-day horizon and turn them into losses.
s_T = s0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)
pnl = s_T - s0
losses = -pnl
alpha = 0.99
var_level = np.quantile(losses, alpha)
es_level = losses[losses >= var_level].mean()

In [ ]:
# Mark the VaR and Expected Shortfall levels on the simulated loss distribution.
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.hist(losses, bins=80, density=True, color="tab:blue", alpha=0.7)
ax.axvline(
    var_level,
    color="tab:red",
    linestyle="--",
    linewidth=1.25,
    label=f"VaR {alpha:.0%}",
)
ax.axvline(
    es_level,
    color="tab:orange",
    linestyle="-.",
    linewidth=1.25,
    label="Expected Shortfall",
)
ax.set_title("Loss Distribution with VaR and Expected Shortfall")
ax.set_xlabel("Loss")
ax.set_ylabel("Density")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

## Figure Generation (Optional)
Run the chapter's figure scripts under `code/figures/` to regenerate the PNG
files under `assets/figures/`.


In [ ]:
import runpy

scripts = [
    "../code/figures/ch13_american_vs_european_put.py",
    "../code/figures/ch13_call_payoffs.py",
    "../code/figures/ch13_cir_euler_hist_and_paths.py",
    "../code/figures/ch13_cir_exact_hist.py",
    "../code/figures/ch13_gbm_paths.py",
    "../code/figures/ch13_gbm_terminal_compare.py",
    "../code/figures/ch13_mc_convergence.py",
    "../code/figures/ch13_random_histograms.py",
    "../code/figures/ch13_random_walk_paths.py",
    "../code/figures/ch13_var_es_hist.py",
]

for script in scripts:
    try:
        runpy.run_path(script, run_name="__main__")
        print(f"OK: {script}")
    except ModuleNotFoundError as e:
        print(f"Skipping {script}: missing dependency ({e.name}).")
    except Exception as e:
        print(f"Failed {script}: {type(e).__name__}: {e}")


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
